# CME Futures: Stochastic Discount-Factor Features

The stochastic discount-factor model learns fold-scoped latent factors from the product panel and
maps them to each declared forward-return horizon. Training rows determine the representation;
validation rows are transformed without refitting. The fitted model, fold identity, prediction
shard, and eligible validation keys are persisted together.

The notebook executes the declared SDF configurations and publishes their catalog rows. IC remains
diagnostic. The equal-weight validation backtest in `13_backtest` selects configurations.

Prerequisites: `03_financial_features`, `04_model_based_features`, and `05_evaluation`.

In [1]:
"""Fit the declared CME futures stochastic discount-factor population."""

import polars as pl

from case_studies.cme_futures.research_workflow import (
    ALL_LABELS,
    model_request_catalog,
    open_study,
    product_universe_table,
    resolve_model_requests,
    resolved_model_plan,
    run_official_model_catalog,
    run_resolved_model_requests,
)

In [2]:
EXECUTION_TIER = "canonical"
WORKSPACE: str | None = None
PREVIEW_REDUCTIONS: dict = {}
# The population hash this run replaces, read from the registry and set by a person. A
# first population takes None; a re-run whose membership has changed is refused without
# the hash it supersedes, and the refusal names the value required.
SUPERSEDES_POPULATION: str | None = None

## Declared requests

Both return horizons use the named stochastic discount-factor configuration. The resolved plan
shows the eligible rows, folds, feature count, checkpoint schedule, and identity before fitting.

**This model publishes on `cuda`, declared in `setup.yaml` rather than detected.** The device
enters both `runtime` and `numerical_runtime` inside the hashed computation, so leaving it to
`preferred_latent_device()` would resolve one training identity on a GPU host and a different one
on a CPU host, both publishing under this population's name. `10a_pca` overrides the same
declaration to `cpu`, because PCA has no GPU implementation to record.

**Nothing here fails on a fit that did not converge.** The shared runner enforces convergence for
IPCA only, and this case study does not declare IPCA; for the stochastic discount factor it
records the training history and the terminal Sharpe as fold extras and checks neither. That is a
gap in shared code rather than in this notebook, so what is claimed below is that every declared
checkpoint was produced and registered - not that the objective had settled when it was.

In [3]:
study = open_study(execution_tier=EXECUTION_TIER, workspace=WORKSPACE)
requests = model_request_catalog(
    "latent_factors",
    labels=ALL_LABELS,
    config_names=("sdf",),
)
resolved = resolve_model_requests(
    study,
    requests,
    execution_tier=EXECUTION_TIER,
    preview_reductions=PREVIEW_REDUCTIONS,
)
universe = product_universe_table()
universe

sector,product,expiry_rule,contract_months
str,str,str,str
"""agriculture""","""ZC""","""business_day_before_15th""","""H,K,N,U,Z"""
"""agriculture""","""ZL""","""business_day_before_15th""","""F,H,K,N,Q,U,V,Z"""
"""agriculture""","""ZM""","""business_day_before_15th""","""F,H,K,N,Q,U,V,Z"""
"""agriculture""","""ZS""","""business_day_before_15th""","""F,H,K,N,Q,U,X"""
"""agriculture""","""ZW""","""business_day_before_15th""","""H,K,N,U,Z"""
…,…,…,…
"""metals""","""SI""","""3rd_last_business_day""","""H,K,N,U,Z"""
"""treasuries""","""ZB""","""last_business_day""","""H,M,U,Z"""
"""treasuries""","""ZF""","""last_business_day""","""H,M,U,Z"""


In [4]:
resolved_model_plan(resolved)

family,label,config_name,task,feature_count,eligible_entities,eligible_rows,folds,validation_start,validation_end,checkpoints,execution_tier,training_hash
str,str,str,str,i64,i64,i64,i64,datetime[μs],datetime[μs],i64,str,str
"""latent_factors""","""fwd_ret_21d""","""sdf""","""regression""",69,30,37782,5,2019-01-03 00:00:00,2023-11-29 00:00:00,5,"""canonical""","""52973301b456"""
"""latent_factors""","""fwd_ret_5d""","""sdf""","""regression""",69,30,38262,5,2019-01-03 00:00:00,2023-12-21 00:00:00,5,"""canonical""","""706e1597b5e6"""


## Execute and validate

The shared latent-factor runner fits each representation inside its training fold, persists the
fitted state, and requires the complete validation key set before publication.

In [5]:
if EXECUTION_TIER == "canonical":
    execution, population = run_official_model_catalog(
        study,
        requests,
        population_name="cme_futures-sdf-validation-v1",
        resolved_requests=resolved,
        supersedes=SUPERSEDES_POPULATION,
    )
else:
    if WORKSPACE is None or not PREVIEW_REDUCTIONS:
        raise ValueError("preview execution requires WORKSPACE and PREVIEW_REDUCTIONS")
    execution = run_resolved_model_requests(study, resolved)
    population = None

Latent factor CV: 1 models × 5 folds
Log file: ~/ml4t/artifacts/case_studies/cme_futures/run_log/training/706e1597b5e6/.models.37b958fb5bc1423ba16a606ae409d609.tmp/latent_factors.log
Scoring: dates=all cadence=- step=1 checkpoint_selection=fixed reporting_epoch=1280
  sdf (K=5):


    Fold 0: ragged train=2059, val=253, max_N=30


      fold 0: reported_epoch=1280, IC=-0.0377, 396.7s


    Fold 1: ragged train=2059, val=258, max_N=30


      fold 1: reported_epoch=1280, IC=+0.0484, 388.3s


...

    Fold 3: ragged train=2043, val=258, max_N=30


      fold 3: reported_epoch=1280, IC=+0.1492, 383.4s


    Fold 4: ragged train=2022, val=258, max_N=30


      fold 4: reported_epoch=1280, IC=+0.0219, 381.3s


    -> best epoch=1280, IC=+0.0368 (1953.5s)
  Best: sdf (IC=+0.0368)


In [6]:
catalog = execution.catalog_rows.select(
    "family",
    "label",
    "config_name",
    "checkpoint_kind",
    "checkpoint_value",
    "execution_tier",
    "complete",
    "training_hash",
    "prediction_hash",
).sort("label", "checkpoint_value")
if catalog.filter(~pl.col("complete")).height:
    raise RuntimeError("stochastic discount-factor execution returned a partial prediction")
catalog

family,label,config_name,checkpoint_kind,checkpoint_value,execution_tier,complete,training_hash,prediction_hash
str,str,str,str,i64,str,bool,str,str
"""latent_factors""","""fwd_ret_21d""","""sdf""","""epoch""",256,"""canonical""",true,"""52973301b456""","""073eadaf2ebd"""
"""latent_factors""","""fwd_ret_21d""","""sdf""","""epoch""",512,"""canonical""",true,"""52973301b456""","""a4d834a6054e"""
"""latent_factors""","""fwd_ret_21d""","""sdf""","""epoch""",768,"""canonical""",true,"""52973301b456""","""63e7b7839194"""
"""latent_factors""","""fwd_ret_21d""","""sdf""","""epoch""",1024,"""canonical""",true,"""52973301b456""","""ea9529f1a791"""
"""latent_factors""","""fwd_ret_21d""","""sdf""","""epoch""",1280,"""canonical""",true,"""52973301b456""","""206874caf483"""
"""latent_factors""","""fwd_ret_5d""","""sdf""","""epoch""",256,"""canonical""",true,"""706e1597b5e6""","""a635e13260db"""
"""latent_factors""","""fwd_ret_5d""","""sdf""","""epoch""",512,"""canonical""",true,"""706e1597b5e6""","""cb9d2e06a0d5"""
"""latent_factors""","""fwd_ret_5d""","""sdf""","""epoch""",768,"""canonical""",true,"""706e1597b5e6""","""4c231319adbd"""
"""latent_factors""","""fwd_ret_5d""","""sdf""","""epoch""",1024,"""canonical""",true,"""706e1597b5e6""","""4805244ef070"""
